In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Metro_Traffic.csv")

df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

X = df.drop(columns=["traffic_volume", "date_time"])
y = df["traffic_volume"]

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (38563, 11)
Testing: (9641, 11)


In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

In [3]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

tree_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

tree_params = {
    "model__max_depth": [5, 10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

tscv = TimeSeriesSplit(n_splits=3)

tree_search = RandomizedSearchCV(
    tree_pipe,
    tree_params,
    n_iter=5,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

tree_search.fit(X_train, y_train)

print("Best tree settings:")
print(tree_search.best_params_)
print("Best CV RMSE:", round(-tree_search.best_score_, 2))

Best tree settings:
{'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_depth': 15}
Best CV RMSE: 580.75


In [4]:
from sklearn.ensemble import RandomForestRegressor

forest_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=1
    ))
])

forest_params = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [10, 15, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

forest_search = RandomizedSearchCV(
    forest_pipe,
    forest_params,
    n_iter=5,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

forest_search.fit(X_train, y_train)

print("Best forest settings:")
print(forest_search.best_params_)
print("Best CV RMSE:", round(-forest_search.best_score_, 2))

Best forest settings:
{'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_depth': 10}
Best CV RMSE: 514.02


In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score

best_models = {
    "Tuned Decision Tree": tree_search.best_estimator_,
    "Tuned Random Forest": forest_search.best_estimator_
}

results = []

for name, model in best_models.items():
    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": round(mean_absolute_error(y_test, pred), 2),
        "RMSE": round(np.sqrt(
            mean_squared_error(y_test, pred)
        ), 2),
        "R2": round(r2_score(y_test, pred), 4)
    })

tuning_results = pd.DataFrame(results)
tuning_results.sort_values("RMSE")